In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
import warnings
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries loaded successfully.')

All libraries loaded successfully.


In [2]:
# ── Load raw CSVs ──────────────────────────────────────────────────────────────
gk = pd.read_csv('all_goalkeepers.csv')
matches = pd.read_csv('matches.csv', low_memory=False)

# Strip BOM from matches column name if present
matches.columns = matches.columns.str.lstrip('\ufeff')

# Trim whitespace from Club abbreviations
gk['Club'] = gk['Club'].str.strip()

print(f'Goalkeepers: {gk.shape[0]} rows × {gk.shape[1]} cols')
print(f'Matches:     {matches.shape[0]} rows × {matches.shape[1]} cols')

gk.head(3)

# ── Filter both datasets to 2012 and later ──────────────────────────────────
gk      = gk[gk['Year'] >= 2012].reset_index(drop=True)
matches = matches[matches['year'] >= 2012].reset_index(drop=True)
print(f'After 2012 filter — Goalkeepers: {gk.shape[0]} rows | Matches: {matches.shape[0]} rows')


FileNotFoundError: [Errno 2] No such file or directory: 'all_goalkeepers.csv'

In [ ]:
# ── Build a club abbreviation → full-name mapping ──────────────────────────────
club_map = {
    'DAL': 'Dallas',
    'MET': 'MetroStars',
    'TB':  'Tampa Bay',
    'LA':  'LA Galaxy',
    'KC':  'KC Wiz',
    'COL': 'Colorado',
    'SJ':  'San Jose',
    'NE':  'New England',
    'DC':  '.',
    'CLB': 'Columbus',
    'CHI': 'Chicago',
    'MIA': 'Miami',
    'RSL': 'Real Salt Lake',
    'ATL': 'Atlanta United',
    'CHV': 'Chivas USA',
    'HOU': 'Houston',
    'NY':  'New York Red Bulls',
    'TOR': 'Toronto FC',
    'SEA': 'Seattle Sounders',
    'NYC': 'NYCFC',
    'POR': 'Portland',
    'VAN': 'Vancouver',
    'MTL': 'Montreal',
    'SKC': 'Sporting KC',
    'ORL': 'Orlando City',
    'NYRB':'New York Red Bulls',
    'MN': 'Minnesota United',
    'LAFC':'LAFC',
    'CIN': 'FC Cincinnati',
    'NAS': 'Nashville SC',
    'IFC': 'Inter Miami',
}

gk['Club_Full'] = gk['Club'].map(club_map).fillna(gk['Club'])
print('Mapping applied. Sample:')
gk[['Club','Club_Full','Year']].drop_duplicates().head(8)

In [ ]:
# ── Basic data-type cleanup ────────────────────────────────────────────────────
# Convert attendance in matches to numeric (has commas)
matches['attendance_num'] = (
    matches['attendance'].astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

# Ensure numeric score columns
matches['home_score'] = pd.to_numeric(matches['home_score'], errors='coerce')
matches['away_score'] = pd.to_numeric(matches['away_score'], errors='coerce')

# Total goals per match
matches['total_goals'] = matches['home_score'] + matches['away_score']

# Win-loss label for GK (1 = winning record, 0 = not)
gk['winning_record'] = (gk['W'] > gk['L']).astype(int)

print('Dtypes cleaned. GK winning_record distribution:')
print(gk['winning_record'].value_counts())

In [ ]:
# ── Missing Value Treatment ──────────────────────────────────────────────────
# Drop rows in goalkeepers where Club or Club_Full is null
# These are players with no club assignment — not useful for analysis
print(f'GK rows before drop: {gk.shape[0]}')
gk = gk.dropna(subset=['Club', 'Club_Full']).reset_index(drop=True)
print(f'GK rows after dropping null Club/Club_Full: {gk.shape[0]}')

# Drop the referee column entirely — 100% missing after 2012 filter
if 'referee' in matches.columns:
   matches = matches.drop(columns=['referee'])
   print('Dropped: referee column (100% missing)')
print(f'Matches rows before processing: {matches.shape[0]}')

# Fill missing attendance with median

attendance_median = matches['attendance_num'].median()

matches['attendance_num'] = matches['attendance_num'].fillna(attendance_median)

print(f'Median attendance used: {attendance_median}')
print(matches['attendance_num'].isnull().sum())

print(f'Matches rows after processing: {matches.shape[0]}')
print(f'Median attendance used: {attendance_median}')

In [ ]:
# ── Load DataFrames into SQLite ────────────────────────────────────────────────
conn = sqlite3.connect(':memory:')
gk.to_sql('goalkeepers', conn, index=False, if_exists='replace')
matches.to_sql('matches', conn, index=False, if_exists='replace')
print('Tables loaded into SQLite:', [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 1 — Basic overview: count rows and seasons in each table.
# WHY: Verify data loaded correctly and understand temporal coverage.
# HOW: Simple COUNT + MIN/MAX aggregation on each table.
# ─────────────────────────────────────────────────────────────────────────────
q1 = pd.read_sql_query("""
    SELECT
        'goalkeepers' AS table_name,
        COUNT(*)        AS total_rows,
        MIN(Year)       AS first_year,
        MAX(Year)       AS last_year,
        COUNT(DISTINCT Club) AS unique_clubs
    FROM goalkeepers
    UNION ALL
    SELECT
        'matches',
        COUNT(*),
        MIN(year),
        MAX(year),
        COUNT(DISTINCT home)
    FROM matches
""", conn)
print('Query 1 — Dataset Overview')
q1

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 2 — GROUP BY: Average goalkeeper metrics by year (regular season only).
# WHY: Identify whether defensive play has improved over MLS history.
# HOW: GROUP BY Year, compute AVG on key performance indicators.
# ─────────────────────────────────────────────────────────────────────────────
q2 = pd.read_sql_query("""
    SELECT
        Year,
        COUNT(*)              AS num_keepers,
        ROUND(AVG(GAA), 3)    AS avg_GAA,
        ROUND(AVG("Sv%"), 2)  AS avg_save_pct,
        ROUND(AVG(ShO), 2)    AS avg_shutouts,
        ROUND(AVG("W%"), 2)   AS avg_win_pct
    FROM goalkeepers
    WHERE Season = 'reg'
    GROUP BY Year
    ORDER BY Year
""", conn)
print('Query 2 — Avg GK Metrics by Year (regular season)')
q2.head(10)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 3 — GROUP BY: Goals allowed and saves per club across all years.
# WHY: Spot which franchises have historically been strong or weak defensively.
# HOW: GROUP BY Club, SUM goals-allowed and saves, compute overall save%.
# ─────────────────────────────────────────────────────────────────────────────
q3 = pd.read_sql_query("""
    SELECT
        Club,
        COUNT(DISTINCT Year)              AS seasons,
        SUM(GA)                           AS total_goals_allowed,
        SUM(SV)                           AS total_saves,
        ROUND(SUM(SV) * 100.0 /
              NULLIF(SUM(SV) + SUM(GA), 0), 2) AS career_save_pct,
        SUM(ShO)                          AS total_shutouts
    FROM goalkeepers
    WHERE Season = 'reg'
    GROUP BY Club
    ORDER BY career_save_pct DESC
""", conn)
print('Query 3 — Career Defensive Stats by Club')
q3

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 4 — JOIN: Attach goalkeeper save% to matches where they were
#           the home or away keeper's club in the same year.
# WHY: Link match-level outcomes to the keeper's season performance.
# HOW: INNER JOIN goalkeepers to matches on Club_Full = home AND Year = year;
#      repeat for away side, then UNION ALL.
# ─────────────────────────────────────────────────────────────────────────────
q4 = pd.read_sql_query("""
    SELECT
        g.Player,
        g.Club,
        g.Year,
        ROUND(g."Sv%", 2)   AS save_pct,
        m.home,
        m.away,
        m.home_score,
        m.away_score,
        'home' AS side
    FROM goalkeepers g
    INNER JOIN matches m
        ON g.Club_Full = m.home
        AND g.Year    = m.year

    UNION ALL

    SELECT
        g.Player,
        g.Club,
        g.Year,
        ROUND(g."Sv%", 2),
        m.home,
        m.away,
        m.home_score,
        m.away_score,
        'away'
    FROM goalkeepers g
    INNER JOIN matches m
        ON g.Club_Full = m.away
        AND g.Year    = m.year
    LIMIT 100
""", conn)
print(f'Query 4 — GK–Match JOIN ({len(q4)} preview rows)')
q4.head(8)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 5 — JOIN: Average attendance per year joined with avg goals per match.
# WHY: Understand whether higher-scoring games drive attendance.
# HOW: Sub-aggregate matches by year, JOIN to goalkeeper year summary.
# ─────────────────────────────────────────────────────────────────────────────
q5 = pd.read_sql_query("""
    SELECT
        m.year,
        ROUND(AVG(m.attendance_num), 0)       AS avg_attendance,
        ROUND(AVG(m.home_score + m.away_score), 2) AS avg_goals_per_match,
        ROUND(g.avg_save_pct, 2)              AS avg_keeper_save_pct
    FROM matches m
    LEFT JOIN (
        SELECT Year, AVG("Sv%") AS avg_save_pct
        FROM goalkeepers
        WHERE Season = 'reg'
        GROUP BY Year
    ) g ON m.year = g.Year
    WHERE m.home_score IS NOT NULL
    GROUP BY m.year
    ORDER BY m.year
""", conn)
print('Query 5 — Attendance × Goals × Save% by Year (JOIN)')
q5.head(10)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 6 — JOIN + aggregate: Top home vs away win rates by club.
# WHY: Quantify home-field advantage in MLS history.
# HOW: JOIN home & away results for each club, compute win rates.
# ─────────────────────────────────────────────────────────────────────────────
q6 = pd.read_sql_query("""
    SELECT
        club,
        SUM(home_games)  AS home_games,
        SUM(home_wins)   AS home_wins,
        ROUND(100.0 * SUM(home_wins) / NULLIF(SUM(home_games), 0), 1) AS home_win_pct,
        SUM(away_games)  AS away_games,
        SUM(away_wins)   AS away_wins,
        ROUND(100.0 * SUM(away_wins) / NULLIF(SUM(away_games), 0), 1) AS away_win_pct
    FROM (
        -- Home side
        SELECT
            home AS club,
            COUNT(*) AS home_games,
            SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) AS home_wins,
            0 AS away_games, 0 AS away_wins
        FROM matches
        WHERE home_score IS NOT NULL
        GROUP BY home

        UNION ALL

        -- Away side
        SELECT
            away AS club,
            0, 0,
            COUNT(*),
            SUM(CASE WHEN away_score > home_score THEN 1 ELSE 0 END)
        FROM matches
        WHERE away_score IS NOT NULL
        GROUP BY away
    )
    GROUP BY club
    HAVING SUM(home_games) >= 20
    ORDER BY home_win_pct DESC
    LIMIT 15
""", conn)
print('Query 6 — Home vs Away Win % by Club')
q6

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 7 — WINDOW FUNCTION: Rank goalkeepers by Save% within each year.
# WHY: Identify the best keeper per season without losing other rows.
# HOW: RANK() OVER (PARTITION BY Year ORDER BY Sv% DESC) — window function.
# ─────────────────────────────────────────────────────────────────────────────
q7 = pd.read_sql_query("""
    SELECT
        Year,
        Player,
        Club,
        ROUND("Sv%", 2)   AS save_pct,
        GP,
        GA,
        ShO,
        RANK() OVER (
            PARTITION BY Year
            ORDER BY "Sv%" DESC
        ) AS season_rank
    FROM goalkeepers
    WHERE Season = 'reg'
      AND GP >= 10
    ORDER BY Year, season_rank
""", conn)

# Show only the #1 keeper each year
best_per_year = q7[q7['season_rank'] == 1].reset_index(drop=True)
print('Query 7 — Best Keeper (by Save%) Each Season (min 10 GP)')
best_per_year

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 8 — WINDOW FUNCTION: Running total of league goals scored by year.
# WHY: Show cumulative offensive output growth as MLS expanded.
# HOW: SUM() OVER (ORDER BY year ROWS UNBOUNDED PRECEDING) — running sum.
# ─────────────────────────────────────────────────────────────────────────────
q8 = pd.read_sql_query("""
    WITH yearly AS (
        SELECT
            year,
            COUNT(*)                           AS num_matches,
            SUM(home_score + away_score)        AS total_goals
        FROM matches
        WHERE home_score IS NOT NULL
        GROUP BY year
    )
    SELECT
        year,
        num_matches,
        total_goals,
        ROUND(CAST(total_goals AS REAL) / num_matches, 2) AS goals_per_match,
        SUM(total_goals) OVER (
            ORDER BY year
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_goals
    FROM yearly
    ORDER BY year
""", conn)
print('Query 8 — Running Total of League Goals by Year (Window Function)')
q8

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 9 — SUBQUERY (inline): Keepers with above-average Save% for their year.
# WHY: Find overachieving keepers relative to the league standard.
# HOW: Inline subquery in WHERE clause computes per-year avg save%.
# ─────────────────────────────────────────────────────────────────────────────
q9 = pd.read_sql_query("""
    SELECT
        g.Player,
        g.Club,
        g.Year,
        ROUND(g."Sv%", 2)        AS save_pct,
        ROUND(avg.avg_sv, 2)     AS league_avg_sv,
        ROUND(g."Sv%" - avg.avg_sv, 2) AS above_avg_by
    FROM goalkeepers g
    JOIN (
        SELECT Year, AVG("Sv%") AS avg_sv
        FROM goalkeepers
        WHERE Season = 'reg' AND GP >= 10
        GROUP BY Year
    ) avg ON g.Year = avg.Year
    WHERE g.Season = 'reg'
      AND g.GP     >= 10
      AND g."Sv%"  > avg.avg_sv
    ORDER BY above_avg_by DESC
    LIMIT 20
""", conn)
print('Query 9 — Keepers with Above-Average Save% (Inline Subquery)')
q9

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 10 — CTE (subquery style): Best career save% among keepers
#            who played ≥5 seasons and ≥100 games.
# WHY: Identify all-time elite keepers on a large enough sample.
# HOW: CTE aggregates career totals; outer query filters and ranks.
# ─────────────────────────────────────────────────────────────────────────────
q10 = pd.read_sql_query("""
    WITH career AS (
        SELECT
            Player,
            COUNT(DISTINCT Year)            AS seasons_played,
            SUM(GP)                         AS career_gp,
            SUM(SV)                         AS career_saves,
            SUM(GA)                         AS career_ga,
            SUM(ShO)                        AS career_shutouts,
            SUM(W)                          AS career_wins,
            ROUND(
                SUM(SV) * 100.0 / NULLIF(SUM(SV) + SUM(GA), 0)
            , 2)                            AS career_save_pct
        FROM goalkeepers
        WHERE Season = 'reg'
        GROUP BY Player
    )
    SELECT *
    FROM career
    WHERE seasons_played >= 5
      AND career_gp      >= 100
    ORDER BY career_save_pct DESC
    LIMIT 20
""", conn)
print('Query 10 — All-Time Best Keepers (CTE, ≥5 seasons, ≥100 GP)')
q10

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 11 — GROUP BY ROLLUP simulation: Goals allowed by Club AND Season type.
# WHY: Compare regular-season vs playoff defensive performance per club.
# HOW: SQLite lacks ROLLUP; we UNION the detail and subtotal levels manually
#      to replicate GROUP BY ROLLUP(Club, Season) behavior.
# ─────────────────────────────────────────────────────────────────────────────
q11 = pd.read_sql_query("""
    -- Detail: Club × Season
    SELECT Club, Season,
           ROUND(AVG(GAA), 3) AS avg_GAA,
           SUM(GA)            AS total_GA,
           COUNT(*)           AS keeper_seasons,
           'detail'           AS rollup_level
    FROM goalkeepers
    GROUP BY Club, Season

    UNION ALL

    -- Subtotal: Club only (Season rolled up)
    SELECT Club, 'ALL',
           ROUND(AVG(GAA), 3),
           SUM(GA),
           COUNT(*),
           'club_subtotal'
    FROM goalkeepers
    GROUP BY Club

    UNION ALL

    -- Grand total
    SELECT 'ALL', 'ALL',
           ROUND(AVG(GAA), 3),
           SUM(GA),
           COUNT(*),
           'grand_total'
    FROM goalkeepers

    ORDER BY Club, Season
""", conn)
print('Query 11 — ROLLUP: Goals Allowed by Club × Season Type')
q11.head(20)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 12 — Window + JOIN: Season rank of each keeper AND their club's
#            home win rate, to see if great keepers correlate with home form.
# WHY: Combine player-level and team-level info in one analytical result.
# HOW: CTE builds home win rates from matches; second CTE ranks keepers;
#      final SELECT JOINs them on Club_Full = home team name.
# ─────────────────────────────────────────────────────────────────────────────
q12 = pd.read_sql_query("""
    WITH home_rates AS (
        SELECT
            home AS club,
            year,
            COUNT(*) AS games,
            ROUND(100.0 * SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END)
                  / COUNT(*), 1) AS home_win_pct
        FROM matches
        WHERE home_score IS NOT NULL
        GROUP BY home, year
    ),
    ranked_keepers AS (
        SELECT
            Player, Club_Full, Year,
            ROUND("Sv%", 2) AS save_pct,
            GAA,
            RANK() OVER (PARTITION BY Year ORDER BY "Sv%" DESC) AS yr_rank
        FROM goalkeepers
        WHERE Season = 'reg' AND GP >= 10
    )
    SELECT
        rk.Player,
        rk.Club_Full,
        rk.Year,
        rk.save_pct,
        rk.GAA,
        rk.yr_rank,
        hr.home_win_pct
    FROM ranked_keepers rk
    LEFT JOIN home_rates hr
        ON rk.Club_Full = hr.club
        AND rk.Year     = hr.year
    WHERE rk.yr_rank <= 5
    ORDER BY rk.Year, rk.yr_rank
""", conn)
print('Query 12 — Top-5 Keepers per Year with Club Home Win % (Window + JOIN)')
q12.head(20)